# Inferred Copy-Number Variation (infercnvpy)

**Approach.**
1. Infer large-scale CNV profiles per cell with infercnvpy, using stromal and immune cells
   as the diploid reference.
2. Cluster cells on their CNV profiles; each CNV cluster gets a CNV score (mean absolute CNV).
3. Call a cell "**Malignant**" when its CNV score is above a threshold **and** its CNV cluster is
   not one of the clusters judged non-malignant (reference-like profile). Optional
   patient-specific thresholds handle low-purity samples.
4. Remaining "Epithelial+Tumor cells" become "Non-malignant epithelial".
5. Estimate intratumour heterogeneity (ITH) from CNV and expression profiles.

**Output.** `cell_type_new` (broad labels with Malignant / Non-malignant epithelial),
CNV matrix and scores, CNV clusters, ITH scores; raw counts in `X`.

Adapted with modification from: https://github.com/icbi-lab/infercnvpy

In [ ]:
# !pip install --quiet scanpy infercnvpy gtfparse
# %cd <project>/notebooks

In [ ]:
import os
import infercnvpy as cnv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

## Config

In [ ]:
input_path = "./../snRNAseq/output/cellbender/cleanup/scVI_integrated_adata_clean_latent20_major_cell_type_add.h5ad"

output_path = "./../snRNAseq/output/cellbender/cleanup/scVI_integrated_adata_clean_latent20_major_cell_type_inferCNV_add.h5ad"

gtf_path = "./../GTF/gencode.v44.annotation.gtf"     # same annotation as the Cell Ranger reference
figures_output_dir = "./../figures-cellbender/integration/4_inferCNV"
tables_output_dir = "./../results/inferCNV"

label_col = "cell_type"                 # broad labels from S14
new_label_col = "cell_type_new"         # output: broad labels with Malignant / Non-malignant epithelial
patient_col = "ID"                      # patient (adjacent normal and tumour share it)
sample_col = "ID_2"                     # patient sample (TN001 vs TN001_AN)
counts_layer = "raw_counts"

## 1. Load, normalise and add genomic positions

infercnvpy works on log-normalised expression; raw counts stay in `layers['raw_counts']`.
Genes without a GTF position are skipped by inferCNV.

In [ ]:
adata = sc.read(input_path)
adata.uns.pop("log1p", None)
adata.X = adata.layers[counts_layer].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

cnv.io.genomic_position_from_gtf(gtf_file=gtf_path, adata=adata, gtf_gene_id="gene_name", inplace=True)
print(f"genes with genomic position: {adata.var['chromosome'].notna().sum()} / {adata.n_vars}")

# --- 1. inferCNV ---
reference_cats = ["Endothelial", "Myeloid", "Lymphoid", "Fibroblast", "Non-adeno"]   # assumed diploid
window_size = 100                       # genes per smoothing window
n_jobs = 8


missing_ref = set(reference_cats) - set(adata.obs[label_col])
assert not missing_ref, f"reference labels not found: {sorted(missing_ref)}"
adata.obs[label_col].value_counts()

## 2. Infer CNV profiles

In [ ]:
cnv.tl.infercnv(adata, reference_key=label_col, reference_cat=reference_cats,
                window_size=window_size, n_jobs=n_jobs)
cnv.tl.pca(adata)
# dendrograms are built on the CNV PCA (scanpy's default would use expression)
sc.tl.dendrogram(adata, groupby=label_col, use_rep="X_cnv_pca")
cnv.pl.chromosome_heatmap(adata, groupby=label_col, dendrogram=True, vmax=heatmap_vmax,
                          save=f"_{label_col}.pdf")

## 3. CNV clusters and CNV score

`cnv_score` is computed **per CNV cluster** (mean absolute CNV of the cluster), so all cells
in a cluster share one value. The cluster table documents the non-malignant cluster choice:
reference-like clusters have low scores and are dominated by reference or normal-tissue labels.

In [ ]:
cnv.pp.neighbors(adata)
cnv.tl.leiden(adata, resolution=cnv_leiden_resolution, random_state=0)
cnv.tl.umap(adata)
cnv.tl.cnv_score(adata)


# --- 2. CNV clusters ---
cnv_leiden_resolution = 1.0
expected_n_cnv_clusters = 39            # guard: the cluster list below is only valid for this clustering
non_malignant_cnv_clusters = ["0", "1", "2", "8", "9", "21", "36"]   # reference-like CNV profiles (from heatmaps)


n_cnv = adata.obs["cnv_leiden"].nunique()
if expected_n_cnv_clusters is not None and n_cnv != expected_n_cnv_clusters:
    raise ValueError(f"{n_cnv} CNV clusters, non_malignant_cnv_clusters was chosen for {expected_n_cnv_clusters}")

sc.tl.dendrogram(adata, groupby="cnv_leiden", use_rep="X_cnv_pca")
cnv.pl.chromosome_heatmap_summary(adata, groupby="cnv_leiden", dendrogram=True, save="_cnv_leiden_summary.pdf")

fig, axs = plt.subplots(1, 3, figsize=(16, 5), gridspec_kw={"wspace": 0.4})
cnv.pl.umap(adata, color="cnv_leiden", legend_loc="on data", legend_fontoutline=2, ax=axs[0], show=False)
cnv.pl.umap(adata, color="cnv_score", color_map="OrRd", ax=axs[1], show=False)
cnv.pl.umap(adata, color=label_col, ax=axs[2], show=False)
plt.savefig(f"{figures_output_dir}/cnv_umap_clusters_score_celltype.pdf", bbox_inches="tight")
plt.show()

obs = adata.obs
cluster_table = pd.DataFrame({
    "n_cells": obs["cnv_leiden"].value_counts(),
    "cnv_score": obs.groupby("cnv_leiden", observed=True)["cnv_score"].first(),
    "frac_reference": obs[label_col].isin(reference_cats).groupby(obs["cnv_leiden"], observed=True).mean().round(2),
    "top_labels": obs.groupby("cnv_leiden", observed=True)[label_col].agg(
        lambda s: "; ".join(f"{k} ({v:.0%})" for k, v in s.value_counts(normalize=True).head(3).items())),
    "n_patients": obs.groupby("cnv_leiden", observed=True)[patient_col].nunique(),
})
cluster_table["non_malignant"] = cluster_table.index.isin(non_malignant_cnv_clusters)
cluster_table.sort_values("cnv_score").to_csv(f"{tables_output_dir}/cnv_cluster_summary.csv")
cluster_table.sort_values("cnv_score")

## 4. Malignant call

In [ ]:
# --- 3. malignant call ---
cnv_score_threshold = 0.009             # cnv_score >= threshold -> highCNV
patient_thresholds = {"R001": 0.0085, "E001": 0.0085}   # low-purity patients re-checked with a lower threshold- did some pre-benchmarking to check
# labels eligible to become Malignant; None reproduces the original (any label, incl. alveolar/airway)
malignant_candidate_labels = None
# e.g. ["Epithelial+Tumor cells", "Alveolar cell type I", "Alveolar cell type II", "Transitional"]
relabel_remaining = {"Epithelial+Tumor cells": "Non-malignant epithelial"}

# CNV score status, with patient-specific thresholds
threshold = obs[patient_col].astype(str).map(patient_thresholds).fillna(cnv_score_threshold)
adata.obs["cnv_status_score"] = np.where(obs["cnv_score"] >= threshold, "highCNV", "lowCNV")
adata.obs["cnv_status_cluster"] = np.where(obs["cnv_leiden"].isin(non_malignant_cnv_clusters),
                                           "Non-malignant_cluster", "Malignant_cluster")

malignant = (adata.obs["cnv_status_score"] == "highCNV") & (adata.obs["cnv_status_cluster"] == "Malignant_cluster")
if malignant_candidate_labels is not None:
    malignant &= obs[label_col].isin(malignant_candidate_labels)

new = obs[label_col].astype(str).replace(relabel_remaining)
new[malignant] = "Malignant"
adata.obs[new_label_col] = new.astype("category")
for col in ["cnv_status_score", "cnv_status_cluster"]:
    adata.obs[col] = adata.obs[col].astype("category")

fig, axs = plt.subplots(1, len(patient_thresholds) + 1, figsize=(4 * (len(patient_thresholds) + 1), 3))
bins = np.linspace(obs["cnv_score"].min(), obs["cnv_score"].max(), 40)
axs[0].hist(obs["cnv_score"], bins=bins, color="grey")
axs[0].axvline(cnv_score_threshold, ls="--", c="red")
axs[0].set_title("all cells")
for ax, (pid, thr) in zip(axs[1:], patient_thresholds.items()):
    ax.hist(obs.loc[obs[patient_col] == pid, "cnv_score"], bins=bins, color="grey")
    ax.axvline(thr, ls="--", c="red")
    ax.set_title(pid)
for ax in axs:
    ax.set_xlabel("CNV score")
plt.tight_layout()
plt.savefig(f"{figures_output_dir}/histogram_cnv_score_thresholds.pdf", bbox_inches="tight")
plt.show()

label_vs_call = pd.crosstab(obs[label_col], adata.obs[new_label_col] == "Malignant").rename(
    columns={True: "Malignant", False: "not malignant"})
label_vs_call.to_csv(f"{tables_output_dir}/malignant_call_by_{label_col}.csv")
label_vs_call

Per-sample malignant fraction. Adjacent-normal samples (`*_AN`) should have few or no
malignant cells; a high fraction there points to a threshold or reference problem.

In [ ]:
per_sample = pd.crosstab(adata.obs[sample_col], adata.obs[new_label_col], normalize="index").mul(100).round(1)
per_sample.to_csv(f"{tables_output_dir}/{new_label_col}_percent_per_{sample_col}.csv")
display(per_sample[["Malignant"]].sort_values("Malignant", ascending=False).T)

sc.pl.umap(adata, color=[new_label_col, "cnv_status_score", "cnv_status_cluster", "cnv_score"],
           color_map="magma_r", ncols=2, wspace=0.4, size=10, save="_malignant_call.pdf")


def stacked_bars(col, fname):
    props = pd.crosstab(adata.obs[sample_col], adata.obs[col], normalize="index").mul(100).iloc[::-1]
    ax = props.plot.barh(stacked=True, edgecolor="none", width=0.85, figsize=(4, 0.25 * len(props) + 1), grid=False)
    ax.set_xlim(0, 100)
    ax.set_xlabel(f"{col} (%)")
    ax.legend(bbox_to_anchor=(1, 1), frameon=False, fontsize=8)
    plt.savefig(f"{figures_output_dir}/{fname}", bbox_inches="tight", dpi=300)
    plt.show()


for col in ["cnv_status_score", "cnv_status_cluster", new_label_col]:
    stacked_bars(col, f"barplot_{sample_col}_proportions_{col}.pdf")

## 5. CNV heatmaps by final label (Normal adjacent control tissues will be your control estimate)

In [ ]:
# --- 4. heatmaps by final label ---
heatmap_sets = {
    "all": None,
    "epithelial": ["Malignant", "Non-malignant epithelial"],
    "non_malignant": ["Myeloid", "Lymphoid", "Fibroblast", "Endothelial", "Alveolar cell type I",
                      "Alveolar cell type II", "Club cell", "Goblet cell", "Transitional", "Non-adeno",
                      "Ciliated epithelial"],
}
heatmap_vmin, heatmap_vmax = -0.2, 0.5

for name, labels in heatmap_sets.items():
    sub = adata if labels is None else adata[adata.obs[new_label_col].isin(labels)]
    if labels is not None:
        missing = set(labels) - set(adata.obs[new_label_col])
        if missing:
            print(f"{name}: labels not present, skipped -> {sorted(missing)}")
    cnv.pl.chromosome_heatmap(sub, groupby=new_label_col, vmin=heatmap_vmin, vmax=heatmap_vmax,
                              figsize=(11, 9 if labels is None else 5), save=f"_{new_label_col}_{name}.pdf")

## 6. Save (X = raw counts)

In [ ]:
adata.X = adata.layers[counts_layer].copy()
adata.uns.pop("log1p", None)
adata.write(output_path)
adata.obs[new_label_col].value_counts()